In [1]:
import os
import sys
import yaml
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if os.path.basename(os.getcwd()) != 'audio_event_detection':
    os.chdir('..')
from models.ast_model import AudioSpectrogramTransformer
from scripts.train import Trainer

from scripts.evaluate import ModelEvaluator
from utils.dataset import AudioEventDataset

In [2]:
# Load tập test
config_path = 'configs/config.yaml'
metadata_df = pd.read_csv('data/processed/spectrograms/processed_metadata.csv')
train_df, temp_df = train_test_split(metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['label'])

with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)
batch_size = config_dict['training']['batch_size']
num_workers = config_dict.get('hardware', {}).get('num_workers', 4)
pin_memory = config_dict.get('hardware', {}).get('pin_memory', True)

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
test_dataset = AudioEventDataset(test_df, config_path, mode='test')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=pin_memory)


In [3]:
best_model_path = os.path.join(os.getcwd(), config_dict['paths']['checkpoint_dir'], 'best_model.pth')
Evaluator = ModelEvaluator(
    model_path=best_model_path
)

Evaluator initialized on cpu


In [4]:
Evaluator.evaluate(test_loader)


Evaluating model...


Evaluation: 100%|██████████| 34/34 [00:58<00:00,  1.72s/it]


{'metrics': {'accuracy': 0.9199255121042831,
  'precision': 0.7832107754697499,
  'recall': 0.6314150725597036,
  'f1_score': 0.6830692644410412,
  'precision_gunshot': np.float64(0.8947368421052632),
  'recall_gunshot': np.float64(0.8947368421052632),
  'f1_gunshot': np.float64(0.8947368421052632),
  'precision_explosion': np.float64(0.5),
  'recall_explosion': np.float64(0.25),
  'f1_explosion': np.float64(0.3333333333333333),
  'precision_siren': np.float64(0.8829787234042553),
  'recall_siren': np.float64(0.8924731182795699),
  'f1_siren': np.float64(0.8877005347593583),
  'precision_glass_breaking': np.float64(0.5),
  'recall_glass_breaking': np.float64(0.25),
  'f1_glass_breaking': np.float64(0.3333333333333333),
  'precision_scream': np.float64(0.7613636363636364),
  'recall_scream': np.float64(0.67),
  'f1_scream': np.float64(0.7127659574468085),
  'precision_dog_bark': np.float64(1.0),
  'recall_dog_bark': np.float64(0.5),
  'f1_dog_bark': np.float64(0.6666666666666666),
  'pr